In [2]:
import re
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer

In [3]:
MODEL_NAME = "HooshvareLab/bert-fa-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
import re

# تعریف الگوها به صورت دیکشنری {label: list_of_patterns}
PATTERN_DICT = {
    'NEG_COMBINED': [
        r'(?:بدون|فاقد)\s+آسانسور\s+و\s+پارکینگ'
    ],
    'ELEVATOR_NEG': [
        r'آسانسور\s+(ندارد|نیست|وجود ندارد)',
        r'بدون\s+آسانسور',
        r'فاقد\s+آسانسور'
    ],
    'PARKING_NEG': [
        r'پارکینگ\s+(ندارد|نیست|وجود ندارد)',
        r'بدون\s+پارکینگ',
        r'فاقد\s+پارکینگ'
    ],
    'ELEVATOR_POS': [
        r'(?:آسانسور|اسانسور)(?:\s*(?:دارد|داره|وجود دارد))?',
        r'دارای\s+آسانسور'
    ],
    'PARKING_POS': [
        r'(?:پارکینگ|پارکینک)(?:\s*(?:دارد|داره|اختصاصی|مسقف))?',
        r'دارای\s+پارکینگ'
    ],
    'REBUILT_POS': [
        r'نوساز\s*(?:کامل)?',
        r'بازسازی\s*(?:شده|کامل)',
        r'کلید\s*نخورده',
        r'صفر\s*کیلومتر'
    ],
    'REBUILT_NEG': [
        r'(قدیمی|تخریبی|کلنگی)'
    ]
}

# ساخت regex واحد با named groups
patterns = []
for label, pattern_list in PATTERN_DICT.items():
    # ترکیب الگوهای یک برچسب با |
    combined_pattern = '|'.join(f'(?:{p})' for p in pattern_list)
    patterns.append(f'(?P<{label}>{combined_pattern})')

combined_regex = re.compile('|'.join(patterns), re.UNICODE)

In [8]:

# ============================================
# توابع کمکی
# ============================================
def find_spans(text):
    spans = []
    for match in combined_regex.finditer(text):
        start, end = match.span()
        for label in PATTERN_DICT.keys():
            if match.group(label):
                spans.append((start, end, label))
                break
    spans.sort(key=lambda x: (x[0], -(x[1]-x[0])))
    return spans

def expand_neg_combined_span(text, start, end):
    """
    برای span با label 'NEG_COMBINED'، دو span جداگانه برای آسانسور و پارکینگ ایجاد می‌کند.
    """
    sub_text = text[start:end]
    spans = []
    # پیدا کردن موقعیت کلمه آسانسور
    elevator_match = re.search(r'آسانسور', sub_text)
    if elevator_match:
        s = start + elevator_match.start()
        e = start + elevator_match.end()
        spans.append((s, e, 'ELEVATOR_NEG'))
    # پیدا کردن موقعیت کلمه پارکینگ
    parking_match = re.search(r'پارکینگ', sub_text)
    if parking_match:
        s = start + parking_match.start()
        e = start + parking_match.end()
        spans.append((s, e, 'PARKING_NEG'))
    return spans

def tokenize_and_label(text):
    encoding = tokenizer(
        text,
        truncation=True,
        max_length=512,
        return_offsets_mapping=True,
        return_tensors=None
    )
    input_ids = encoding['input_ids']
    offset_mapping = encoding['offset_mapping']
    labels = ['O'] * len(input_ids)

    spans = find_spans(text)
    # تبدیل spans ترکیبی به چندین span ساده
    expanded_spans = []
    for s, e, label in spans:
        if label == 'NEG_COMBINED':
            expanded_spans.extend(expand_neg_combined_span(text, s, e))
        else:
            expanded_spans.append((s, e, label))
    # حذف همپوشانی‌ها (اولویت با span بلندتر)
    expanded_spans.sort(key=lambda x: (x[0], -(x[1]-x[0])))
    token_used = set()
    for start_char, end_char, label in expanded_spans:
        first_token = last_token = None
        for idx, (tok_start, tok_end) in enumerate(offset_mapping):
            if tok_start == 0 and tok_end == 0:
                continue
            if tok_start <= start_char < tok_end and first_token is None:
                first_token = idx
            if tok_start < end_char <= tok_end and last_token is None:
                last_token = idx
        if first_token is not None and last_token is not None:
            for idx in range(first_token, last_token+1):
                if idx in token_used:
                    continue
                if idx == first_token:
                    labels[idx] = f"B-{label}"
                else:
                    labels[idx] = f"I-{label}"
                token_used.add(idx)
    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    return tokens, labels

def process_csv_to_conll(input_csv, output_conll, text_column='full_text'):
    df = pd.read_csv(input_csv)
    print(f"📂 بارگذاری {len(df)} رکورد از {input_csv}")
    with open(output_conll, 'w', encoding='utf-8') as f:
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
            text = row[text_column]
            if not isinstance(text, str) or len(text.strip()) < 10:
                f.write("\n")
                continue
            try:
                tokens, labels = tokenize_and_label(text)
                filtered = [(t, l) for t, l in zip(tokens, labels) if t not in ['[CLS]', '[SEP]', '[PAD]']]
                for token, label in filtered:
                    f.write(f"{token}\t{label}\n")
                f.write("\n")
            except Exception as e:
                print(f"⚠️ خطا در ردیف {idx}: {e}")
                f.write("\n")
    print(f"✅ فایل CoNLL ذخیره شد: {output_conll}")


In [10]:

# مستقیماً در کولب
input_file = "/content/ner_data_final.csv"
output_file = "ner_data_final.conll"
process_csv_to_conll(input_file, output_file, text_column='full_text')

📂 بارگذاری 19724 رکورد از /content/ner_data_final.csv


Processing: 100%|██████████| 19724/19724 [00:14<00:00, 1393.28it/s]

✅ فایل CoNLL ذخیره شد: ner_data_final.conll


In [11]:
from collections import Counter
import re

def analyze_conll(file_path, num_sentences=10):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    sentences = []
    current = []
    for line in lines:
        if line.strip() == "":
            if current:
                sentences.append(current)
                current = []
        else:
            parts = line.strip().split('\t')
            if len(parts) == 2:
                current.append(parts)

    print(f"📊 تعداد جملات: {len(sentences)}")

    # توزیع برچسب‌ها (غیر O)
    label_counter = Counter()
    for sent in sentences:
        for _, label in sent:
            if label != 'O':
                label_counter[label] += 1

    print("\n📈 توزیع برچسب‌ها (غیر O):")
    for label, count in label_counter.most_common():
        print(f"  {label}: {count}")

    # نمایش چند جمله نمونه که حاوی برچسب‌های خاص هستند
    print("\n🔍 نمونه جملات با برچسب‌های منفی:")
    neg_labels = {'B-ELEVATOR_NEG', 'I-ELEVATOR_NEG', 'B-PARKING_NEG', 'I-PARKING_NEG', 'B-REBUILT_NEG', 'I-REBUILT_NEG'}
    shown = 0
    for sent in sentences:
        sent_labels = [label for _, label in sent if label in neg_labels]
        if sent_labels:
            text = ' '.join([tok for tok, _ in sent[:30]])
            print(f"\n  جمله: {text[:150]}...")
            print(f"  برچسب‌ها: {sent_labels[:10]}")
            shown += 1
            if shown >= num_sentences:
                break

    # بررسی نفی ترکیبی
    print("\n🔎 جستجوی نفی ترکیبی ('بدون آسانسور و پارکینگ'):")
    combined_neg_count = 0
    for sent in sentences:
        text = ' '.join([tok for tok, _ in sent])
        if 'بدون' in text and 'آسانسور' in text and 'پارکینگ' in text:
            combined_neg_count += 1
            if combined_neg_count <= 3:
                print(f"\n  مثال {combined_neg_count}: {text[:200]}...")
    print(f"\n✅ تعداد کل جملات با نفی ترکیبی: {combined_neg_count}")

analyze_conll("/content/ner_data_final.conll", num_sentences=5)

📊 تعداد جملات: 19724

📈 توزیع برچسب‌ها (غیر O):
  B-PARKING_POS: 11132
  B-REBUILT_POS: 8701
  I-REBUILT_POS: 8572
  B-ELEVATOR_POS: 7415
  I-PARKING_POS: 2560
  I-PARKING_NEG: 2099
  B-PARKING_NEG: 2098
  B-REBUILT_NEG: 1863
  I-ELEVATOR_POS: 1002
  B-ELEVATOR_NEG: 876
  I-ELEVATOR_NEG: 868
  I-REBUILT_NEG: 75

🔍 نمونه جملات با برچسب‌های منفی:

  جمله: اپارتمان ۷۰ متری اندیشه گلستان ۱ ۷۰ متر تک خوابه دوکل ##ه و غرق در نور طبقه ۲ بدون اسانسور دارای پارکینگ اختصاصی هر طبقه ۲ واحد کل ساختمان ۷...
  برچسب‌ها: ['B-ELEVATOR_NEG', 'I-ELEVATOR_NEG']

  جمله: اپارتمان مسکن مهر ۸۰ ##متری در ایوان دارای درب ضدسر ##قت هال بزرگتر نسبت به سایر واحدهای مسکن مهر ( به دلیل اضافه کردن یک اتاق به هال ) ویو...
  برچسب‌ها: ['B-ELEVATOR_NEG', 'I-ELEVATOR_NEG']

  جمله: زعفرانیه [UNK] تا [UNK] پروژه های ستاره دار - مجری فروش پروژه های خاص مسکونی - [UNK] تا [UNK] متر اپارتمانهای لاکچری [UNK] شمیرانات و لواسانات [UNK] م...
  برچسب‌ها: ['B-REBUILT_NEG']

  جمله: اپارتمان ، [UNK] / دوکل ##ه / فول / ben ##ya ##m

training

In [1]:
!pip install -q transformers datasets seqeval accelerate evaluate


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.6 MB/s eta 0:00:00


In [7]:
# !pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00


In [2]:
import evaluate
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset, DatasetDict
# import evaluate
from collections import Counter

In [3]:

# ============================================
# 3. بارگذاری فایل CoNLL
# ============================================
def read_conll(file_path):
    """خواندن فایل CoNLL و برگرداندن لیست توکن‌ها و برچسب‌ها"""
    tokens = []
    labels = []
    with open(file_path, 'r', encoding='utf-8') as f:
        cur_tokens = []
        cur_labels = []
        for line in f:
            if line.strip() == "":
                if cur_tokens:
                    tokens.append(cur_tokens)
                    labels.append(cur_labels)
                    cur_tokens = []
                    cur_labels = []
            else:
                parts = line.strip().split('\t')
                if len(parts) == 2:
                    cur_tokens.append(parts[0])
                    cur_labels.append(parts[1])
        if cur_tokens:
            tokens.append(cur_tokens)
            labels.append(cur_labels)
    return {"tokens": tokens, "ner_tags": labels}

data = read_conll("/content/ner_data_final.conll")
dataset = Dataset.from_dict(data)

# تقسیم داده (80% train, 10% val, 10% test)
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_val = split_dataset["train"]
test_dataset = split_dataset["test"]
split_val = train_val.train_test_split(test_size=0.1, seed=42)  # 10% of 80% = 8% val, 72% train
train_dataset = split_val["train"]
val_dataset = split_val["test"]

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")


Train: 14201, Val: 1578, Test: 3945


In [4]:

# ============================================
# 4. توکنایزر و لیست برچسب‌ها
# ============================================
model_name = "HooshvareLab/bert-fa-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# لیست برچسب‌های یکتا
all_labels = sorted(set([label for sent in data["ner_tags"] for label in sent]))
label2id = {l: i for i, l in enumerate(all_labels)}
id2label = {i: l for l, i in label2id.items()}
num_labels = len(all_labels)
print(f"تعداد برچسب‌ها: {num_labels}")
print("برچسب‌ها:", all_labels)

# ============================================
# 5. تابع توکن‌سازی و هم‌راستایی
# ============================================
def tokenize_and_align(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=512,
        padding="max_length"
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                # برای توکن‌های بعدی کلمه، اگر برچسب O است -100، در غیر این صورت همان برچسب
                label_ids.append(label2id[label[word_idx]] if label[word_idx] != "O" else -100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_train = train_dataset.map(tokenize_and_align, batched=True)
tokenized_val = val_dataset.map(tokenize_and_align, batched=True)
tokenized_test = test_dataset.map(tokenize_and_align, batched=True)

# ============================================
# 6. محاسبه وزن کلاس‌ها (class_weight)
# ============================================
# شمارش تعداد هر برچسب در داده آموزش
label_counts = Counter()
for labels in tokenized_train["labels"]:
    for lbl in labels:
        if lbl != -100:
            label_counts[lbl] += 1

total = sum(label_counts.values())
class_weights = {idx: total / (len(label_counts) * count) for idx, count in label_counts.items()}
# تبدیل به tensor برای استفاده در loss
class_weights_tensor = torch.tensor([class_weights.get(i, 1.0) for i in range(num_labels)], dtype=torch.float)

# بارگذاری مدل با تنظیم class_weight
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)
# تنظیم وزن کلاس‌ها در loss
model.config.problem_type = "single_label_classification"
model.config.label2id = label2id
model.config.id2label = id2label
# برای BCEWithLogitsLoss نیست، برای cross-entropy باید وزن را به loss_fn بدهیم
# اما Trainer از loss_fn مدل استفاده می‌کند – نیاز به override ندارد چون cross-entropy از weight قبول می‌کند.
# روش ساده: مدل را با class_weight در forward اصلاح کنیم – اما بهتر است از یک Trainer سفارشی استفاده کنیم.

# # روش ساده: ایجاد Trainer سفارشی با تغییر compute_loss
# class WeightedTrainer(Trainer):
#     def __init__(self, class_weights, *args, **kwargs):
#         super().__init__(*args, **kwargs)
#         self.class_weights = class_weights.to(self.model.device)

#     def compute_loss(self, model, inputs, return_outputs=False):
#         labels = inputs.pop("labels")
#         outputs = model(**inputs)
#         logits = outputs.logits
#         loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights, ignore_index=-100)
#         loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
#         return (loss, outputs) if return_outputs else loss


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/440 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

تعداد برچسب‌ها: 13
برچسب‌ها: ['B-ELEVATOR_NEG', 'B-ELEVATOR_POS', 'B-PARKING_NEG', 'B-PARKING_POS', 'B-REBUILT_NEG', 'B-REBUILT_POS', 'I-ELEVATOR_NEG', 'I-ELEVATOR_POS', 'I-PARKING_NEG', 'I-PARKING_POS', 'I-REBUILT_NEG', 'I-REBUILT_POS', 'O']


Map:   0%|          | 0/14201 [00:00<?, ? examples/s]

Map:   0%|          | 0/1578 [00:00<?, ? examples/s]

Map:   0%|          | 0/3945 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/654M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: HooshvareLab/bert-fa-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:

In [5]:
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights.to(self.model.device)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):   # اضافه کردن num_items_in_batch
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights, ignore_index=-100)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [6]:

# ============================================
# 7. تابع محاسبه متریک با seqeval
# ============================================
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []
    for pred, label in zip(predictions, labels):
        pred_ids = []
        label_ids = []
        for p_id, l_id in zip(pred, label):
            if l_id != -100:
                pred_ids.append(id2label[p_id])
                label_ids.append(id2label[l_id])
        true_predictions.append(pred_ids)
        true_labels.append(label_ids)

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# ============================================
# 8. تنظیمات آموزش (با لاگ‌گیری دقیق)
# ============================================
training_args = TrainingArguments(
    output_dir="./ner_model_improved",
    eval_strategy="steps",
    eval_steps=200,                 # هر 200 استپ ارزیابی انجام شود
    save_strategy="steps",
    save_steps=200,
    logging_steps=50,               # هر 50 استپ train_loss چاپ شود
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    report_to=["none"],            # برای جلوگیری از ارسال به external services
)

data_collator = DataCollatorForTokenClassification(tokenizer)

# استفاده از WeightedTrainer
trainer = WeightedTrainer(
    class_weights=class_weights_tensor,
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [7]:

# ============================================
# 9. شروع آموزش
# ============================================
print("🚀 شروع آموزش...")
trainer.train()


🚀 شروع آموزش...


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
200,0.428088,0.335403,0.157866,0.866136,0.267056,0.895955
400,0.255026,0.404109,0.244963,0.880546,0.383295,0.944354
600,0.166776,0.300855,0.336354,0.886614,0.487693,0.963395
800,0.249734,0.192335,0.473279,0.863102,0.611335,0.979131
1000,0.116347,0.198543,0.438528,0.890027,0.587558,0.976061
1200,0.148987,0.157081,0.382304,0.868411,0.530891,0.971002


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [12]:
from transformers import AutoModelForTokenClassification
import torch

# ============================================
# بارگذاری مدل از checkpoint-800
# ============================================
checkpoint_path = "/content/ner_model_improved/checkpoint-800"
model = AutoModelForTokenClassification.from_pretrained(checkpoint_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)  # اگر دستگاه (cuda/cpu) تعریف شده است، در غیر این صورت حذف کنید

# اگر trainer از قبل تعریف شده، می‌توانید مدل آن را جایگزین کنید
trainer.model = model

# ============================================
# ارزیابی روی مجموعه تست
# ============================================
print("\n📊 ارزیابی روی مجموعه تست با checkpoint-800:")
test_results = trainer.evaluate(tokenized_test)
for key, value in test_results.items():
    print(f"  {key}: {value:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


📊 ارزیابی روی مجموعه تست با checkpoint-800:


Step,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
200,0.428088,0.335403,0.157866,0.866136,0.267056,0.895955
400,0.255026,0.404109,0.244963,0.880546,0.383295,0.944354
600,0.166776,0.300855,0.336354,0.886614,0.487693,0.963395
800,0.249734,0.192335,0.473279,0.863102,0.611335,0.979131
1000,0.116347,0.198543,0.438528,0.890027,0.587558,0.976061
1200,0.148987,0.157081,0.382304,0.868411,0.530891,0.971002
1280,0.118822,0.141665,0.480250,0.880116,0.621415,0.979486


  eval_loss: 0.1417
  eval_precision: 0.4803
  eval_recall: 0.8801
  eval_f1: 0.6214
  eval_accuracy: 0.9795


In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

# ============================================
# بارگذاری مدل نهایی (مسیر را تنظیم کنید)
# ============================================
model_path = "/content/ner_model_improved/checkpoint-800"  # یا مسیر checkpoint مورد نظر
tokenizer = AutoTokenizer.from_pretrained("HooshvareLab/bert-fa-base-uncased")
model = AutoModelForTokenClassification.from_pretrained(model_path)
model.eval()

# ============================================
# تابع پیش‌بینی
# ============================================
def predict_entities(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=2)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    labels = [model.config.id2label[p.item()] for p in predictions[0]]

    result = []
    for tok, lab in zip(tokens, labels):
        if tok not in ["[CLS]", "[SEP]", "[PAD]"] and lab != "O":
            result.append((tok, lab))
    return result

# ============================================
# ۵ جمله تست (متنوع از نظر مثبت، منفی، نفی ترکیبی)
# ============================================
# test_sentences = [
#     "آپارتمان ۸۵ متری دوخوابه با پارکینگ و آسانسور، بازسازی کامل",
#     "واحد ۷۵ متری بدون آسانسور، اما پارکینگ دارد",
#     "ساختمان ۱۲۰ متری بدون آسانسور و پارکینگ، قدیمی و کلنگی",
#     "آپارتمان ۱۰۰ متری فول آپشن در منطقه خوب",
#     "آسانسور ندارد، پارکینگ نیست"
# ]

test_sentences = [
    "آپارتمان ۸۰ متری دو خوابه با آسانسور و پارکینگ",
    "واحد ۱۰۰ متری سه خواب، آسانسور ندارد، پارکینگ دارد",
    "ساختمان ۶۰ متری قدیمی، بدون آسانسور، نیاز به بازسازی",
    "آپارتمان ۱۲۰ متری نوساز با پارکینگ اختصاصی",
    "فروش آپارتمان ۹۰ متری در منطقه خوب",
    "واحد ۷۰ متری پارکینگ ندارد، آسانسور دارد",
    "آپارتمان ۵۰ متری بازسازی شده، فاقد پارکینگ",
    "ساختمان ۱۵۰ متری کلنگی، بدون پارکینگ",
    "آپارتمان ۱۳۰ متری دارای آسانسور و پارکینگ و انباری",
    "واحد ۸۵ متری اسانسور دارد، پارکینک ندارد",
    "آپارتمان ۹۵ متری دوخواب، پارکینگ نیست، آسانسور هست",
    "ساختمان ۱۱۰ متری سه خواب، آسانسور دارد، پارکینگ ندارد",
    "آپارتمان ۷۵ متری کلید نخورده، با پارکینگ",
    "واحد ۶۵ متری یک خواب، بدون آسانسور، بدون پارکینگ",
    "آپارتمان ۱۰۵ متری بازسازی نشده، قدیمی",
    "فروش واحد ۴۰ متری استودیویی، فاقد آسانسور",
    "آپارتمان ۱۱۵ متری با پارکینگ مسقف و آسانسور برند",
    "ساختمان ۲۰۰ متری ویلایی، بدون آسانسور، پارکینگ دارد",
    "آپارتمان ۸۸ متری نوساز کامل، آسانسور ندارد",
    "واحد ۹۲ متری تخریبی، نیاز به بازسازی اساسی"
]

# ============================================
# اجرا و نمایش نتایج
# ============================================
print("🧪 نتایج پیش‌بینی مدل NER:\n")
for i, sent in enumerate(test_sentences, 1):
    print(f"{i}. متن: {sent}")
    entities = predict_entities(sent, model, tokenizer)
    if entities:
        for tok, lab in entities:
            print(f"      {tok:15} -> {lab}")
    else:
        print("      هیچ موجودیتی یافت نشد.")
    print("-" * 50)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🧪 نتایج پیش‌بینی مدل NER:

1. متن: آپارتمان ۸۰ متری دو خوابه با آسانسور و پارکینگ
      اسانسور         -> B-ELEVATOR_POS
      پارکینگ         -> B-PARKING_POS
--------------------------------------------------
2. متن: واحد ۱۰۰ متری سه خواب، آسانسور ندارد، پارکینگ دارد
      اسانسور         -> B-ELEVATOR_NEG
      ندارد           -> I-ELEVATOR_NEG
      پارکینگ         -> B-PARKING_POS
      دارد            -> I-PARKING_POS
--------------------------------------------------
3. متن: ساختمان ۶۰ متری قدیمی، بدون آسانسور، نیاز به بازسازی
      قدیمی           -> B-REBUILT_NEG
      بدون            -> B-ELEVATOR_NEG
      اسانسور         -> I-ELEVATOR_NEG
      بازسازی         -> B-REBUILT_POS
--------------------------------------------------
4. متن: آپارتمان ۱۲۰ متری نوساز با پارکینگ اختصاصی
      نوساز           -> B-REBUILT_POS
      پارکینگ         -> B-PARKING_POS
      اختصاصی         -> I-PARKING_POS
--------------------------------------------------
5. متن: فروش آپارتمان ۹۰ متری د

In [13]:
import zipfile
import os
from google.colab import files

# مسیر فایل‌ها
checkpoint_dir = "/content/ner_model_improved/checkpoint-800"
files_to_zip = [
    os.path.join(checkpoint_dir, "config.json"),
    os.path.join(checkpoint_dir, "model.safetensors")
]

# ایجاد فایل زیپ
zip_path = "/content/model_checkpoint_800.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in files_to_zip:
        if os.path.exists(file):
            zipf.write(file, arcname=os.path.basename(file))
        else:
            print(f"⚠️ فایل یافت نشد: {file}")



In [14]:
# دانلود زیپ
files.download(zip_path)
print("✅ فایل model_checkpoint_800.zip دانلود شد.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ فایل model_checkpoint_800.zip دانلود شد.


In [17]:
test_sentences = [
    "آپارتمان ۸۰ متری دو خوابه با آسانسور و پارکینگ",
    "واحد ۱۰۰ متری سه خواب، آسانسور ندارد، پارکینگ دارد",
    "ساختمان ۶۰ متری قدیمی، بدون آسانسور، نیاز به بازسازی",
    "آپارتمان ۱۲۰ متری نوساز با پارکینگ اختصاصی",
    "فروش آپارتمان ۹۰ متری در منطقه خوب",
    "واحد ۷۰ متری پارکینگ ندارد، آسانسور دارد",
    "آپارتمان ۵۰ متری بازسازی شده، فاقد پارکینگ",
    "ساختمان ۱۵۰ متری کلنگی، بدون پارکینگ",
    "آپارتمان ۱۳۰ متری دارای آسانسور و پارکینگ و انباری",
    "واحد ۸۵ متری اسانسور دارد، پارکینک ندارد",
    "آپارتمان ۹۵ متری دوخواب، پارکینگ نیست، آسانسور هست",
    "ساختمان ۱۱۰ متری سه خواب، آسانسور دارد، پارکینگ ندارد",
    "آپارتمان ۷۵ متری کلید نخورده، با پارکینگ",
    "واحد ۶۵ متری یک خواب، بدون آسانسور، بدون پارکینگ",
    "آپارتمان ۱۰۵ متری بازسازی نشده، قدیمی",
    "فروش واحد ۴۰ متری استودیویی، فاقد آسانسور",
    "آپارتمان ۱۱۵ متری با پارکینگ مسقف و آسانسور برند",
    "ساختمان ۲۰۰ متری ویلایی، بدون آسانسور، پارکینگ دارد",
    "آپارتمان ۸۸ متری نوساز کامل، آسانسور ندارد",
    "واحد ۹۲ متری تخریبی، نیاز به بازسازی اساسی"
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)  # اگر دستگاه (cuda/cpu) تعریف شده است، در غیر این صورت حذف کنید


BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(100000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, e

In [18]:
print("🧪 نتایج پیش‌بینی مدل NER:\n")
for i, sent in enumerate(test_sentences, 1):
    print(f"{i}. متن: {sent}")
    entities = predict_entities(sent, model, tokenizer)
    if entities:
        for tok, lab in entities:
            print(f"      {tok:15} -> {lab}")
    else:
        print("      هیچ موجودیتی یافت نشد.")
    print("-" * 50)

🧪 نتایج پیش‌بینی مدل NER:

1. متن: آپارتمان ۸۰ متری دو خوابه با آسانسور و پارکینگ


RuntimeError: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)